# STAT 4830 Final Project Demo (Colab)

This notebook mirrors the practical report layout and reproduces key tables/figures from saved artifacts.

## Sections
1. Executive summary
2. Pipeline diagrams
3. Baseline results + architecture testing
4. Online A/B gate results
5. Return trajectories vs benchmarks
6. IPO weight trajectory
7. Validation-loss diagnostics
8. 12-day IPO validation case
9. Commodities case
10. Lessons learned
11. Future work

In [ ]:
from pathlib import Path
import subprocess
import re
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

ROOT = Path('/content/STAT-4830-OSO')
if not ROOT.exists() and Path('/content').exists():
    subprocess.run([
        'git', 'clone', 'https://github.com/shivani9298/STAT-4830-OSO.git', str(ROOT)
    ], check=False)

if not ROOT.exists():
    ROOT = Path.cwd()

print('Using ROOT:', ROOT)

In [ ]:
def parse_summary(path: Path):
    if not path.exists():
        return None
    txt = path.read_text()
    def grab(pattern, cast=float, default=None):
        m = re.search(pattern, txt)
        return cast(m.group(1)) if m else default
    return {
        'total_return_pct': grab(r'Total return:\s+([\-\d\.]+)%'),
        'ann_return_pct': grab(r'Return \(annualized\):\s+([\-\d\.]+)%'),
        'ann_vol_pct': grab(r'Volatility \(annual\):\s+([\-\d\.]+)%'),
        'sharpe': grab(r'Sharpe \(annualized\):\s+([\-\d\.]+)'),
        'max_dd_pct': grab(r'Max drawdown:\s+([\-\d\.]+)%'),
    }

summary_offline = parse_summary(ROOT / 'results' / 'ipo_optimizer_summary.txt')
summary_val12 = parse_summary(ROOT / 'results' / 'ipo_optimizer_summary_val12_adaptive.txt')
summary_commodity = parse_summary(ROOT / 'results' / 'commodity_optimizer_summary.txt')

for name, obj in [('offline', summary_offline), ('val12', summary_val12), ('commodity', summary_commodity)]:
    if obj is None:
        print(f'[warning] missing summary file for: {name}')

display(Markdown('## 1) Executive Summary'))
exec_bullets = [
    'Built a risk-aware GRU allocator (offline + online update variants).',
    'Online cadence/confidence gating improved process control but did not beat high-IPO static benchmarks in IPO-led regimes.',
    '12-day validation window was unstable for selection; scheduler wiring and validation split design materially affect conclusions.'
]
for b in exec_bullets:
    display(Markdown(f'- {b}'))

In [ ]:
display(Markdown('## 2) Data + Model Pipeline Diagram'))
for p in [
    ROOT / 'figures' / 'architecture' / 'offline_gru_static_pipeline_visual.png',
    ROOT / 'figures' / 'architecture' / 'online_gru_adaptive_pipeline_clean.png',
]:
    if p.exists():
        display(Markdown(f'**{p.name}**'))
        display(Image(filename=str(p)))
    else:
        display(Markdown(f'Missing: `{p}`'))

In [ ]:
display(Markdown('## 3) Baseline Results Table (Model vs Market vs IPO vs 50/50)'))

baseline_table = pd.DataFrame([
    ['Model', 87.52, 97.97, 19.26, 3.65, -7.20],
    ['Market only', 19.40, 21.24, 12.25, 1.63, -7.89],
    ['IPO only', 192.78, 221.19, 31.03, 3.92, -10.08],
    ['Equal 50/50', 88.52, 99.11, 19.39, 3.65, -7.20],
], columns=['Strategy', 'Total Return %', 'Ann Return %', 'Ann Vol %', 'Sharpe', 'Max DD %'])

display(baseline_table)

display(Markdown('### Architecture Testing (Transformer, GRU, LSTM)'))
arch_table = pd.DataFrame([
    ['GRU', 'Core default model', 'Robust in full pipeline', 'Primary model'],
    ['LSTM', 'Matched-run metrics file', 'Competitive with GRU', 'Fallback'],
    ['Transformer', 'Ablation JSON', 'Sensitive/less consistent', 'Not primary'],
], columns=['Architecture', 'Evidence Source', 'Performance Signal', 'Decision'])
display(arch_table)

if (ROOT / 'results' / 'ipo_optimizer_gru_lstm_metrics_w126.txt').exists():
    print((ROOT / 'results' / 'ipo_optimizer_gru_lstm_metrics_w126.txt').read_text())

for p in [
    ROOT / 'figures' / 'ipo_optimizer_gru_vs_lstm_loss_w126.png',
    ROOT / 'figures' / 'ipo_optimizer' / 'comparison' / 'gru_vs_lstm_val_loss.png',
    ROOT / 'figures' / 'ipo_optimizer' / 'comparison' / 'gru_vs_lstm_validation_returns_side_by_side.png',
    ROOT / 'figures' / 'online_evaluation' / 'val_loss_hypothesis_experiments.png',
]:
    if p.exists():
        display(Markdown(f'**{p.name}**'))
        display(Image(filename=str(p)))

In [ ]:
display(Markdown('## 4) Online A/B Gate Results'))
ab_path = ROOT / 'results' / 'ab_update_gate_results.csv'
if ab_path.exists():
    ab = pd.read_csv(ab_path)
    display(ab)
else:
    display(Markdown(f'Missing: `{ab_path}`'))
img = ROOT / 'figures' / 'online_evaluation' / 'ab_update_gate_results.png'
if img.exists():
    display(Image(filename=str(img)))
else:
    display(Markdown(f'Missing: `{img}`'))

In [ ]:
display(Markdown('## 5) Return Trajectories (Cadence/Confidence/50-50 + Market + IPO)'))
img = ROOT / 'figures' / 'online_evaluation' / 'online_return_trajectories_all_benchmarks.png'
if img.exists():
    display(Image(filename=str(img)))
else:
    display(Markdown(f'Missing: `{img}`'))

cad_path = ROOT / 'results' / 'online_path_cadence_lb504.csv'
con_path = ROOT / 'results' / 'online_path_confidence_lb504.csv'
if cad_path.exists() and con_path.exists():
    cad = pd.read_csv(cad_path)
    con = pd.read_csv(con_path)
    rm = cad['realized_market'].to_numpy()
    ri = cad['realized_ipo'].to_numpy()
    ret = {
        'Cadence gate (net)': cad['net_ret'].to_numpy(),
        'Confidence gate (net)': con['net_ret'].to_numpy(),
        'Static 50/50': 0.5 * rm + 0.5 * ri,
        'Market-only': rm,
        'IPO-only': ri,
    }
    for k, v in ret.items():
        final = (pd.Series(v).add(1).prod() - 1) * 100
        print(f'{k}: {final:.2f}%')
else:
    display(Markdown(f'Missing required paths: `{cad_path}` and/or `{con_path}`'))

In [ ]:
display(Markdown('## 6) IPO Weight Trajectory (Online Underweighting Evidence)'))
img = ROOT / 'figures' / 'online_evaluation' / 'online_ipo_weight_trajectory.png'
if img.exists():
    display(Image(filename=str(img)))

In [ ]:
display(Markdown('## 7) Validation-Loss Diagnostics'))
h_path = ROOT / 'results' / 'val_loss_hypothesis_experiments.csv'
if h_path.exists():
    df_h = pd.read_csv(h_path)
    display(df_h)
else:
    display(Markdown(f'Missing: `{h_path}`'))
img = ROOT / 'figures' / 'online_evaluation' / 'val_loss_hypothesis_experiments.png'
if img.exists():
    display(Image(filename=str(img)))
else:
    display(Markdown(f'Missing: `{img}`'))

In [ ]:
display(Markdown('## 8) 12-Day Validation Window Case Study (IPO)'))

for p in [
    ROOT / 'figures' / 'ipo_optimizer_replots' / 'loss_semilogy_gru_offline_true_val12_adaptive.png',
    ROOT / 'figures' / 'ipo_optimizer_replots' / 'returns_val12_adaptive_vs_5050.png',
    ROOT / 'figures' / 'ipo_optimizer_replots' / 'ipo_weights_val12_adaptive.png',
]:
    if p.exists():
        display(Markdown(f'**{p.name}**'))
        display(Image(filename=str(p)))

if (ROOT / 'results' / 'ipo_optimizer_summary_val12_adaptive.txt').exists():
    print((ROOT / 'results' / 'ipo_optimizer_summary_val12_adaptive.txt').read_text())

In [ ]:
display(Markdown('## 9) Commodities Case Study'))
if (ROOT / 'results' / 'commodity_optimizer_summary.txt').exists():
    print((ROOT / 'results' / 'commodity_optimizer_summary.txt').read_text())
for p in [
    ROOT / 'figures' / 'commodity_optimizer' / 'gru' / 'validation_returns_vs_equal_weight.png',
    ROOT / 'figures' / 'commodity_optimizer' / 'gru' / 'weight_evolution_over_time.png',
]:
    if p.exists():
        display(Image(filename=str(p)))

In [ ]:
## 10) Lessons Learned / Limitations

- Update gates changed update frequency but did not fix objective/exposure mismatch by themselves.
- Validation design (size and chronology) materially changes conclusions.
- Small 12-day validation windows are useful stress tests, but unstable for model selection.
- Transformer complexity was not consistently rewarded in realized path quality.

## 11) Future Work

- Standardize adaptive scheduler parameters in all training entrypoints.
- Use larger walk-forward validation blocks for robust selection.
- Add richer IPO-entry features and regime indicators.
- Refine online gates with forward utility proxies and explicit exposure constraints.

In [ ]:
display(Markdown('## Dynamic Demo: Live-Style Return Reveal'))
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import numpy as np

def _cum(r):
    return (np.cumprod(1 + np.asarray(r, dtype=float)) - 1) * 100.0

def _animate_returns(dates, series_map, title, interval=60, max_frames=220):
    n = len(dates)
    if n <= 1:
        display(Markdown(f'Not enough data to animate: **{title}**'))
        return

    # Keep animation lightweight for Colab by sampling frames if long.
    if n > max_frames:
        idx = np.linspace(0, n - 1, max_frames).astype(int)
    else:
        idx = np.arange(n)

    fig, ax = plt.subplots(figsize=(10.5, 4.8))
    colors = {
        'Trained model': '#1f77b4',
        'Static 50/50': '#2ca02c',
        'Market-only': '#ff7f0e',
        'IPO-only': '#9467bd',
    }
    styles = {
        'Trained model': '-',
        'Static 50/50': '--',
        'Market-only': '-.',
        'IPO-only': ':',
    }

    lines = {}
    for name, vals in series_map.items():
        (ln,) = ax.plot([], [], label=name, color=colors.get(name, None), ls=styles.get(name, '-'), lw=2)
        lines[name] = ln

    ax.axhline(0, color='k', ls=':', lw=0.8)
    ax.set_title(title)
    ax.set_ylabel('Cumulative Return (%)')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left', fontsize=9)

    all_y = np.concatenate([np.asarray(v, dtype=float) for v in series_map.values()])
    ypad = max(2.0, 0.08 * (all_y.max() - all_y.min() + 1e-9))
    ax.set_ylim(all_y.min() - ypad, all_y.max() + ypad)
    ax.set_xlim(dates.iloc[0], dates.iloc[-1])

    def init():
        for ln in lines.values():
            ln.set_data([], [])
        return list(lines.values())

    def update(frame_i):
        k = idx[frame_i]
        x = dates.iloc[: k + 1]
        for name, vals in series_map.items():
            lines[name].set_data(x, vals[: k + 1])
        return list(lines.values())

    anim = FuncAnimation(
        fig,
        update,
        init_func=init,
        frames=len(idx),
        interval=interval,
        blit=True,
        repeat=False,
    )
    plt.close(fig)
    display(HTML(anim.to_jshtml()))

# --- Long validation window (online path) ---
long_path = ROOT / 'results' / 'online_path_cadence_lb504.csv'
if long_path.exists():
    long_df = pd.read_csv(long_path, parse_dates=['date']).sort_values('date')
    rm = long_df['realized_market'].to_numpy(float)
    ri = long_df['realized_ipo'].to_numpy(float)
    series_long = {
        'Trained model': _cum(long_df['net_ret'].to_numpy(float)),
        'Static 50/50': _cum(0.5 * rm + 0.5 * ri),
        'Market-only': _cum(rm),
        'IPO-only': _cum(ri),
    }
    _animate_returns(
        long_df['date'],
        series_long,
        title='Live Reveal: Long Validation Window Returns',
        interval=45,
        max_frames=240,
    )
else:
    display(Markdown(f'Missing: `{long_path}`'))

# --- 12-day validation window (adaptive LR run) ---
val12_w = ROOT / 'results' / 'ipo_optimizer_weights_val12_adaptive.csv'
val12_src = ROOT / 'results' / 'ipo_optimizer_online_path.csv'
if val12_w.exists() and val12_src.exists():
    w = pd.read_csv(val12_w, parse_dates=['date']).sort_values('date')
    r = pd.read_csv(val12_src, parse_dates=['date']).sort_values('date')
    base = r[['date', 'realized_market', 'realized_ipo']].drop_duplicates('date')
    d = w.merge(base, on='date', how='left')
    if d[['realized_market', 'realized_ipo']].isna().any().any():
        display(Markdown('Some 12-day dates are missing realized returns; skipping animation.'))
    else:
        wm = d['weight_market'].to_numpy(float)
        wi = d['weight_IPO'].to_numpy(float)
        rm = d['realized_market'].to_numpy(float)
        ri = d['realized_ipo'].to_numpy(float)
        model = wm * rm + wi * ri
        series_12 = {
            'Trained model': _cum(model),
            'Static 50/50': _cum(0.5 * rm + 0.5 * ri),
            'Market-only': _cum(rm),
            'IPO-only': _cum(ri),
        }
        _animate_returns(
            d['date'],
            series_12,
            title='Live Reveal: 12-Day Validation Window Returns',
            interval=350,
            max_frames=120,
        )
else:
    display(Markdown(f'Missing: `{val12_w}` and/or `{val12_src}`'))